# Spectral Graph Theory - Worked Examples

This notebook provides 12 comprehensive worked examples covering spectral graph theory concepts essential for understanding Graph Neural Networks (GNNs) and graph-based machine learning.

**Topics Covered:**
1. Graph Laplacian
2. Normalized Laplacian
3. Eigenvalues of Graphs
4. Algebraic Connectivity (Fiedler Value)
5. Spectral Clustering
6. Random Walks on Graphs
7. Graph Fourier Transform
8. Graph Signal Processing Basics
9. Cheeger Inequality
10. Spectral Embedding
11. Graph Partitioning
12. Commute Time and Resistance Distance

**Prerequisites:** Linear algebra, basic graph theory, Python with NumPy, SciPy, NetworkX, and Matplotlib.

In [ ]:
import numpy as np
import scipy.linalg as la
from scipy.sparse.linalg import eigsh
import networkx as nx
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict

np.set_printoptions(precision=4, suppress=True)
plt.style.use('seaborn-v0_8-whitegrid')

---
## Example 1: Graph Laplacian

The **Graph Laplacian** is a fundamental matrix representation that captures the structure of a graph.

### Theory

For an undirected graph $G = (V, E)$ with $n$ vertices:

- **Adjacency Matrix** $A$: $A_{ij} = 1$ if edge $(i,j) \in E$, else $0$
- **Degree Matrix** $D$: Diagonal matrix where $D_{ii} = \sum_j A_{ij}$
- **Laplacian Matrix** $L = D - A$

### Properties
1. $L$ is symmetric and positive semi-definite
2. Row sums are zero: $L \cdot \mathbf{1} = \mathbf{0}$
3. Smallest eigenvalue is always 0
4. **Quadratic Form**: $\mathbf{x}^T L \mathbf{x} = \sum_{(i,j) \in E} (x_i - x_j)^2$ (Dirichlet energy)

In [ ]:
def compute_graph_laplacian(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute the graph Laplacian from adjacency matrix.
    
    Returns: (L, D, A) - Laplacian, Degree matrix, Adjacency
    """
    A = A.astype(float)
    degrees = A.sum(axis=1)
    D = np.diag(degrees)
    L = D - A
    return L, D, A

# Create a sample graph: a kite shape
#     0
#    /|\
#   1-+-2
#    \|/
#     3
#     |
#     4

A = np.array([
    [0, 1, 1, 1, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 1, 0],
    [1, 1, 1, 0, 1],
    [0, 0, 0, 1, 0]
])

L, D, _ = compute_graph_laplacian(A)

print("Adjacency Matrix A:")
print(A)
print("\nDegree Matrix D (diagonal values):")
print(np.diag(D))
print("\nLaplacian Matrix L = D - A:")
print(L)

# Verify properties
print("\n--- Laplacian Properties ---")
print(f"Symmetric: {np.allclose(L, L.T)}")
print(f"Row sums (should be 0): {L.sum(axis=1)}")

# Dirichlet energy example
x = np.array([1, 0.8, 0.9, 0.5, 0.1])  # Smooth signal
energy = x.T @ L @ x
print(f"\nDirichlet energy for smooth signal: {energy:.4f}")

y = np.array([1, -1, 1, -1, 1])  # Oscillating signal
energy_y = y.T @ L @ y
print(f"Dirichlet energy for oscillating signal: {energy_y:.4f}")

---
## Example 2: Normalized Laplacian

The normalized Laplacian normalizes by node degrees, making it more suitable for irregular graphs.

### Two Forms

1. **Symmetric Normalized Laplacian**:
   $$L_{sym} = D^{-1/2} L D^{-1/2} = I - D^{-1/2} A D^{-1/2}$$

2. **Random Walk Laplacian**:
   $$L_{rw} = D^{-1} L = I - D^{-1} A$$

### Properties
- Both have eigenvalues in $[0, 2]$
- Both share the same eigenvalues (but different eigenvectors)
- $L_{rw}$'s eigenvectors relate to random walk dynamics

In [ ]:
def compute_normalized_laplacians(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute both normalized Laplacians.
    
    Returns: (L_sym, L_rw, P) where P is random walk transition matrix
    """
    n = len(A)
    degrees = A.sum(axis=1)
    D = np.diag(degrees)
    L = D - A
    
    # Handle zero degrees
    degrees_safe = np.where(degrees > 0, degrees, 1)
    
    # Symmetric normalized Laplacian
    D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees_safe))
    L_sym = D_inv_sqrt @ L @ D_inv_sqrt
    
    # Random walk Laplacian
    D_inv = np.diag(1.0 / degrees_safe)
    L_rw = np.eye(n) - D_inv @ A
    
    # Transition matrix
    P = D_inv @ A
    
    return L_sym, L_rw, P

# Use our kite graph
L_sym, L_rw, P = compute_normalized_laplacians(A)

print("Symmetric Normalized Laplacian L_sym:")
print(L_sym)

print("\nRandom Walk Laplacian L_rw:")
print(L_rw)

print("\nTransition Matrix P = D^(-1)A:")
print(P)

# Compare eigenvalues
eig_sym = np.sort(la.eigvalsh(L_sym))
eig_rw = np.sort(la.eigvals(L_rw).real)

print("\n--- Eigenvalue Comparison ---")
print(f"L_sym eigenvalues: {eig_sym}")
print(f"L_rw eigenvalues:  {eig_rw}")
print(f"Both in [0, 2]: {all(0 <= e <= 2 for e in eig_sym)}")

---
## Example 3: Eigenvalues of Graphs

The eigenvalues of graph matrices encode important structural information.

### Key Facts

For Laplacian $L$ with eigenvalues $0 = \lambda_1 \le \lambda_2 \le \ldots \le \lambda_n$:

- **Multiplicity of 0**: Equals the number of connected components
- **Largest eigenvalue**: $\lambda_n \le 2 \cdot d_{max}$ (for unnormalized)
- **Spectral gap**: $\lambda_2 - \lambda_1 = \lambda_2$ indicates connectivity strength

### Special Graphs
- **Complete graph $K_n$**: $\{0, n, n, \ldots, n\}$
- **Cycle $C_n$**: $\lambda_k = 2 - 2\cos(2\pi k/n)$
- **Path $P_n$**: $\lambda_k = 2 - 2\cos(\pi k/n)$

In [ ]:
def analyze_graph_spectrum(G: nx.Graph, name: str):
    """
    Compute and display the Laplacian spectrum of a graph.
    """
    A = nx.adjacency_matrix(G).toarray().astype(float)
    L, _, _ = compute_graph_laplacian(A)
    eigenvalues = np.sort(la.eigvalsh(L))
    
    print(f"\n=== {name} (n={G.number_of_nodes()}) ===")
    print(f"Eigenvalues: {eigenvalues}")
    print(f"Algebraic connectivity (λ₂): {eigenvalues[1]:.4f}")
    print(f"Spectral radius (λ_n): {eigenvalues[-1]:.4f}")
    return eigenvalues

# Create different graph types
n = 6

# Complete graph K_n
K_n = nx.complete_graph(n)
eig_complete = analyze_graph_spectrum(K_n, f"Complete Graph K_{n}")

# Cycle graph C_n
C_n = nx.cycle_graph(n)
eig_cycle = analyze_graph_spectrum(C_n, f"Cycle Graph C_{n}")

# Verify analytical formula for cycle
analytical_cycle = [2 - 2*np.cos(2*np.pi*k/n) for k in range(n)]
print(f"\nAnalytical (C_n): {np.sort(analytical_cycle)}")

# Path graph P_n
P_n = nx.path_graph(n)
eig_path = analyze_graph_spectrum(P_n, f"Path Graph P_{n}")

# Star graph S_n
S_n = nx.star_graph(n-1)  # n-1 leaves + 1 center = n nodes
eig_star = analyze_graph_spectrum(S_n, f"Star Graph S_{n}")

# Visualize spectra
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
graphs = [(eig_complete, f'Complete K_{n}'), (eig_cycle, f'Cycle C_{n}'), 
          (eig_path, f'Path P_{n}'), (eig_star, f'Star S_{n}')]

for ax, (eigs, title) in zip(axes, graphs):
    ax.stem(range(len(eigs)), eigs, basefmt=' ')
    ax.set_title(title)
    ax.set_xlabel('Index')
    ax.set_ylabel('λ')
plt.tight_layout()
plt.show()

---
## Example 4: Algebraic Connectivity (Fiedler Value)

The **algebraic connectivity** $\lambda_2$ (Fiedler value) measures how well-connected a graph is.

### Theory

- $\lambda_2 > 0$ iff graph is connected
- Larger $\lambda_2$ = more robust connectivity
- The corresponding eigenvector $v_2$ (Fiedler vector) reveals graph structure

### Bounds
- $\lambda_2 \le \frac{n}{n-1} \cdot \kappa(G)$ where $\kappa(G)$ is vertex connectivity
- For a path: $\lambda_2 = 2(1 - \cos(\pi/n)) \approx \pi^2/n^2$

In [ ]:
def fiedler_analysis(A: np.ndarray, labels: List[str] = None):
    """
    Analyze algebraic connectivity and Fiedler vector.
    """
    L, _, _ = compute_graph_laplacian(A)
    eigenvalues, eigenvectors = la.eigh(L)
    idx = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    lambda_2 = eigenvalues[1]
    fiedler_vector = eigenvectors[:, 1]
    
    n = len(A)
    if labels is None:
        labels = [str(i) for i in range(n)]
    
    print(f"Algebraic Connectivity λ₂ = {lambda_2:.4f}")
    print(f"\nFiedler Vector:")
    for i, (label, val) in enumerate(zip(labels, fiedler_vector)):
        partition = '+' if val > 0 else '-'
        print(f"  Node {label}: {val:+.4f} [{partition}]")
    
    return lambda_2, fiedler_vector

# Create a barbell graph (two cliques connected by a bridge)
G_barbell = nx.barbell_graph(4, 1)  # Two K4 connected by a path of 1
A_barbell = nx.adjacency_matrix(G_barbell).toarray().astype(float)

print("=== Barbell Graph Analysis ===")
print("Two cliques K4 connected by a single edge\n")
lambda2_barbell, fiedler_barbell = fiedler_analysis(A_barbell)

# Compare with more connected graph
print("\n=== Complete Graph K8 ===")
G_complete = nx.complete_graph(8)
A_complete = nx.adjacency_matrix(G_complete).toarray().astype(float)
lambda2_complete, _ = fiedler_analysis(A_complete)

print(f"\nComparison:")
print(f"  Barbell λ₂ = {lambda2_barbell:.4f} (weak connection)")
print(f"  Complete λ₂ = {lambda2_complete:.4f} (strong connection)")

# Visualize Fiedler vector on graph
fig, ax = plt.subplots(figsize=(8, 4))
pos = nx.spring_layout(G_barbell, seed=42)
colors = ['red' if v < 0 else 'blue' for v in fiedler_barbell]
nx.draw(G_barbell, pos, node_color=colors, with_labels=True, 
        node_size=500, font_color='white', ax=ax)
ax.set_title('Barbell Graph - Fiedler Vector Partition (Red: negative, Blue: positive)')
plt.show()

---
## Example 5: Spectral Clustering

**Spectral clustering** uses the Laplacian eigenvectors to find clusters in data or graphs.

### Algorithm

1. Compute the normalized Laplacian $L_{sym}$
2. Find the $k$ smallest eigenvectors $U \in \mathbb{R}^{n \times k}$
3. Normalize rows of $U$
4. Apply k-means to the rows of $U$

### Why It Works
The eigenvectors embed nodes into a space where clustering is easier. The Fiedler vector naturally separates loosely connected parts.

In [ ]:
def spectral_clustering(A: np.ndarray, k: int) -> np.ndarray:
    """
    Perform spectral clustering.
    
    Args:
        A: Adjacency matrix
        k: Number of clusters
    
    Returns:
        Cluster labels
    """
    n = len(A)
    L_sym, _, _ = compute_normalized_laplacians(A)
    
    # Get k smallest eigenvectors
    eigenvalues, eigenvectors = la.eigh(L_sym)
    idx = np.argsort(eigenvalues)
    U = eigenvectors[:, idx[:k]]
    
    # Normalize rows
    row_norms = np.linalg.norm(U, axis=1, keepdims=True) + 1e-10
    U_normalized = U / row_norms
    
    # Simple k-means
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(U_normalized)
    
    return labels, U

# Create graph with clear cluster structure (two communities)
np.random.seed(42)
n1, n2 = 10, 10
n = n1 + n2

# High connectivity within clusters, low between
A_cluster = np.zeros((n, n))

# Cluster 1 (nodes 0-9): dense
for i in range(n1):
    for j in range(i+1, n1):
        if np.random.rand() < 0.7:
            A_cluster[i, j] = A_cluster[j, i] = 1

# Cluster 2 (nodes 10-19): dense
for i in range(n1, n):
    for j in range(i+1, n):
        if np.random.rand() < 0.7:
            A_cluster[i, j] = A_cluster[j, i] = 1

# Few edges between clusters
A_cluster[0, n1] = A_cluster[n1, 0] = 1
A_cluster[n1-1, n-1] = A_cluster[n-1, n1-1] = 1

# Perform spectral clustering
labels, embedding = spectral_clustering(A_cluster, k=2)
true_labels = np.array([0]*n1 + [1]*n2)

print("=== Spectral Clustering Results ===")
print(f"Predicted labels: {labels}")
print(f"True labels:      {true_labels}")

# Handle label permutation
accuracy = max(np.mean(labels == true_labels), np.mean(1-labels == true_labels))
print(f"\nAccuracy: {accuracy:.1%}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

G_cluster = nx.from_numpy_array(A_cluster)
pos = nx.spring_layout(G_cluster, seed=42)

# Ground truth
colors_true = ['red' if l == 0 else 'blue' for l in true_labels]
nx.draw(G_cluster, pos, node_color=colors_true, with_labels=True, 
        node_size=300, ax=axes[0])
axes[0].set_title('Ground Truth Clusters')

# Spectral clustering result
colors_pred = ['red' if l == 0 else 'blue' for l in labels]
nx.draw(G_cluster, pos, node_color=colors_pred, with_labels=True, 
        node_size=300, ax=axes[1])
axes[1].set_title('Spectral Clustering Result')

plt.tight_layout()
plt.show()

---
## Example 6: Random Walks on Graphs

A **random walk** on a graph transitions from node $i$ to neighbor $j$ with probability $P_{ij} = A_{ij}/d_i$.

### Connection to Laplacian

- Transition matrix: $P = D^{-1}A$
- $L_{rw} = I - P$
- Eigenvalues of $P$ are $1 - \lambda_i(L_{rw})$

### Key Properties
- **Stationary distribution**: $\pi_i = d_i / 2m$ (for connected, undirected graphs)
- **Mixing time**: Related to $\lambda_2$ of $L_{rw}$
- Larger spectral gap = faster mixing

In [ ]:
def random_walk_analysis(A: np.ndarray, start_node: int = 0, steps: int = 100):
    """
    Analyze random walk on a graph.
    """
    n = len(A)
    degrees = A.sum(axis=1)
    total_edges = degrees.sum()  # 2m
    
    # Transition matrix
    D_inv = np.diag(1.0 / np.where(degrees > 0, degrees, 1))
    P = D_inv @ A
    
    # Stationary distribution (for undirected graphs)
    pi_stationary = degrees / total_edges
    
    print("=== Random Walk Analysis ===")
    print(f"Transition Matrix P:")
    print(P)
    print(f"\nStationary Distribution π: {pi_stationary}")
    
    # Simulate random walk
    np.random.seed(42)
    current = start_node
    visit_counts = np.zeros(n)
    trajectory = [current]
    
    for _ in range(steps):
        visit_counts[current] += 1
        # Move to random neighbor
        probs = P[current]
        current = np.random.choice(n, p=probs)
        trajectory.append(current)
    
    empirical_dist = visit_counts / steps
    
    print(f"\nEmpirical distribution ({steps} steps from node {start_node}):")
    print(f"  {empirical_dist}")
    print(f"\nConvergence to stationary: {np.allclose(empirical_dist, pi_stationary, atol=0.1)}")
    
    # Distribution over time
    p_t = np.zeros(n)
    p_t[start_node] = 1.0
    
    distributions = [p_t.copy()]
    for t in range(20):
        p_t = P.T @ p_t
        distributions.append(p_t.copy())
    
    return P, pi_stationary, np.array(distributions)

# Use a simple cycle graph
G_cycle = nx.cycle_graph(5)
A_cycle = nx.adjacency_matrix(G_cycle).toarray().astype(float)

P, pi, distributions = random_walk_analysis(A_cycle, start_node=0, steps=1000)

# Visualize distribution evolution
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(5):
    ax.plot(distributions[:, i], label=f'Node {i}', marker='o', markersize=3)
ax.axhline(y=0.2, color='black', linestyle='--', label='Stationary')
ax.set_xlabel('Time step')
ax.set_ylabel('Probability')
ax.set_title('Random Walk Distribution Evolution on C₅')
ax.legend()
plt.show()

---
## Example 7: Graph Fourier Transform

The **Graph Fourier Transform (GFT)** generalizes the classical Fourier transform to signals on graphs.

### Theory

For Laplacian $L = U \Lambda U^T$ (eigendecomposition):

- **GFT**: $\hat{x} = U^T x$ (transform to spectral domain)
- **Inverse GFT**: $x = U \hat{x}$ (transform back)

### Interpretation
- Small eigenvalues → low frequencies (smooth variations)
- Large eigenvalues → high frequencies (rapid variations)
- **Parseval's theorem**: $\|x\|^2 = \|\hat{x}\|^2$

In [ ]:
def graph_fourier_transform(L: np.ndarray, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute Graph Fourier Transform.
    
    Returns: (x_hat, eigenvalues, U)
    """
    eigenvalues, U = la.eigh(L)
    idx = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[idx]
    U = U[:, idx]
    
    x_hat = U.T @ x  # GFT
    return x_hat, eigenvalues, U

def inverse_gft(U: np.ndarray, x_hat: np.ndarray) -> np.ndarray:
    """Inverse Graph Fourier Transform."""
    return U @ x_hat

# Create a path graph (analogous to 1D signal)
n = 16
G_path = nx.path_graph(n)
A_path = nx.adjacency_matrix(G_path).toarray().astype(float)
L_path, _, _ = compute_graph_laplacian(A_path)

# Create different signals
# 1. Smooth signal (low frequency)
x_smooth = np.sin(np.linspace(0, np.pi, n))

# 2. High frequency signal
x_high = np.array([1 if i % 2 == 0 else -1 for i in range(n)]).astype(float)

# 3. Localized impulse
x_impulse = np.zeros(n)
x_impulse[n//2] = 1.0

# Compute GFT for each
x_hat_smooth, eigenvalues, U = graph_fourier_transform(L_path, x_smooth)
x_hat_high, _, _ = graph_fourier_transform(L_path, x_high)
x_hat_impulse, _, _ = graph_fourier_transform(L_path, x_impulse)

# Visualize
fig, axes = plt.subplots(3, 2, figsize=(12, 9))

signals = [(x_smooth, x_hat_smooth, 'Smooth Signal'),
           (x_high, x_hat_high, 'High Frequency Signal'),
           (x_impulse, x_hat_impulse, 'Impulse Signal')]

for idx, (x, x_hat, title) in enumerate(signals):
    # Vertex domain
    axes[idx, 0].stem(range(n), x, basefmt=' ')
    axes[idx, 0].set_title(f'{title} - Vertex Domain')
    axes[idx, 0].set_xlabel('Node')
    
    # Spectral domain
    axes[idx, 1].stem(eigenvalues, np.abs(x_hat)**2, basefmt=' ')
    axes[idx, 1].set_title(f'{title} - Spectral Domain (Power)')
    axes[idx, 1].set_xlabel('Eigenvalue (frequency)')

plt.tight_layout()
plt.show()

# Verify Parseval's theorem
print("=== Parseval's Theorem ===")
for x, x_hat, name in signals:
    energy_vertex = np.sum(x**2)
    energy_spectral = np.sum(x_hat**2)
    print(f"{name}: ||x||² = {energy_vertex:.4f}, ||x̂||² = {energy_spectral:.4f}")

---
## Example 8: Graph Signal Processing Basics

**Graph Signal Processing** applies filtering operations to signals defined on graphs.

### Spectral Filtering

A filter $g(\cdot)$ in the spectral domain:
$$y = g(L)x = U \cdot \text{diag}(g(\lambda_1), \ldots, g(\lambda_n)) \cdot U^T x$$

### Common Filters
- **Low-pass**: $g(\lambda) = e^{-\alpha\lambda}$ (smoothing)
- **High-pass**: $g(\lambda) = 1 - e^{-\alpha\lambda}$ (edge detection)
- **Band-pass**: $g(\lambda) = e^{-(\lambda - \lambda_0)^2/\sigma^2}$

In [ ]:
def spectral_filter(L: np.ndarray, x: np.ndarray, g: callable) -> np.ndarray:
    """
    Apply spectral filter g to signal x on graph with Laplacian L.
    
    Args:
        L: Laplacian matrix
        x: Input signal
        g: Filter function g(λ)
    
    Returns:
        Filtered signal
    """
    eigenvalues, U = la.eigh(L)
    
    # Apply filter in spectral domain
    x_hat = U.T @ x
    g_lambda = np.array([g(lam) for lam in eigenvalues])
    y_hat = g_lambda * x_hat
    
    return U @ y_hat

# Create graph and noisy signal
np.random.seed(42)
n = 20
G = nx.grid_2d_graph(4, 5)  # 4x5 grid
G = nx.convert_node_labels_to_integers(G)
A = nx.adjacency_matrix(G).toarray().astype(float)
L, _, _ = compute_graph_laplacian(A)

# Create smooth signal + noise
x_clean = np.sin(np.linspace(0, 2*np.pi, n))
noise = 0.5 * np.random.randn(n)
x_noisy = x_clean + noise

# Define filters
def low_pass(lam, alpha=2.0):
    return np.exp(-alpha * lam)

def high_pass(lam, alpha=2.0):
    return 1 - np.exp(-alpha * lam)

def band_pass(lam, center=1.0, width=0.5):
    return np.exp(-((lam - center)**2) / (2 * width**2))

# Apply filters
x_lowpass = spectral_filter(L, x_noisy, lambda l: low_pass(l, 1.5))
x_highpass = spectral_filter(L, x_noisy, lambda l: high_pass(l, 1.5))

print("=== Graph Signal Filtering ===")
print(f"Original signal MSE to clean: {np.mean((x_noisy - x_clean)**2):.4f}")
print(f"Low-pass filtered MSE to clean: {np.mean((x_lowpass - x_clean)**2):.4f}")

# Visualize filter responses
eigenvalues_L = np.sort(la.eigvalsh(L))
lam_range = np.linspace(0, eigenvalues_L.max(), 100)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Filter responses
axes[0, 0].plot(lam_range, [low_pass(l, 1.5) for l in lam_range], label='Low-pass')
axes[0, 0].plot(lam_range, [high_pass(l, 1.5) for l in lam_range], label='High-pass')
axes[0, 0].plot(lam_range, [band_pass(l, 2.0, 0.5) for l in lam_range], label='Band-pass')
axes[0, 0].set_xlabel('Eigenvalue λ')
axes[0, 0].set_ylabel('g(λ)')
axes[0, 0].set_title('Filter Frequency Responses')
axes[0, 0].legend()

# Signals
axes[0, 1].plot(x_clean, 'g-', label='Clean', linewidth=2)
axes[0, 1].plot(x_noisy, 'r.', label='Noisy', alpha=0.6)
axes[0, 1].set_title('Original Signals')
axes[0, 1].legend()

axes[1, 0].plot(x_clean, 'g-', label='Clean', linewidth=2)
axes[1, 0].plot(x_lowpass, 'b-', label='Low-pass filtered', linewidth=2)
axes[1, 0].set_title('Low-pass Filtering (Denoising)')
axes[1, 0].legend()

axes[1, 1].plot(x_highpass, 'r-', label='High-pass filtered', linewidth=2)
axes[1, 1].set_title('High-pass Filtering (Edge Detection)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

---
## Example 9: Cheeger Inequality

The **Cheeger inequality** connects the algebraic connectivity $\lambda_2$ to the graph's expansion properties.

### Cheeger Constant (Isoperimetric Number)

$$h(G) = \min_{S \subseteq V, |S| \le n/2} \frac{|\partial S|}{|S|}$$

where $|\partial S|$ is the number of edges between $S$ and $V \setminus S$.

### Cheeger Inequality

$$\frac{\lambda_2}{2} \le h(G) \le \sqrt{2\lambda_2}$$

This bounds the minimum cut in terms of spectral properties!

In [ ]:
from itertools import combinations

def cheeger_constant(A: np.ndarray) -> Tuple[float, set]:
    """
    Compute Cheeger constant by brute force (only for small graphs).
    
    Returns: (h, best_S) - Cheeger constant and the subset achieving it
    """
    n = len(A)
    min_ratio = float('inf')
    best_S = None
    
    # Check all subsets of size 1 to n/2
    for size in range(1, n//2 + 1):
        for S in combinations(range(n), size):
            S_set = set(S)
            complement = set(range(n)) - S_set
            
            # Count boundary edges
            boundary = sum(A[i, j] for i in S for j in complement)
            
            ratio = boundary / len(S)
            if ratio < min_ratio:
                min_ratio = ratio
                best_S = S_set
    
    return min_ratio, best_S

def verify_cheeger_inequality(A: np.ndarray):
    """
    Verify Cheeger inequality for a graph.
    """
    L, _, _ = compute_graph_laplacian(A)
    eigenvalues = np.sort(la.eigvalsh(L))
    lambda_2 = eigenvalues[1]
    
    h, S = cheeger_constant(A)
    
    lower_bound = lambda_2 / 2
    upper_bound = np.sqrt(2 * lambda_2)
    
    print("=== Cheeger Inequality Verification ===")
    print(f"Algebraic connectivity λ₂ = {lambda_2:.4f}")
    print(f"Cheeger constant h(G) = {h:.4f}")
    print(f"Minimum cut subset S = {S}")
    print(f"\nCheeger bounds: {lower_bound:.4f} ≤ h(G) ≤ {upper_bound:.4f}")
    print(f"Actual h(G) = {h:.4f}")
    print(f"Inequality satisfied: {lower_bound <= h <= upper_bound}")
    
    return lambda_2, h, S

# Test on different graphs
# 1. Barbell graph (should have low Cheeger constant)
G_barbell = nx.barbell_graph(4, 0)
A_barbell = nx.adjacency_matrix(G_barbell).toarray().astype(float)
print("\n--- Barbell Graph (weak connection) ---")
verify_cheeger_inequality(A_barbell)

# 2. Complete graph (high Cheeger constant)
G_complete = nx.complete_graph(6)
A_complete = nx.adjacency_matrix(G_complete).toarray().astype(float)
print("\n--- Complete Graph K_6 (strong connection) ---")
verify_cheeger_inequality(A_complete)

# 3. Cycle graph (intermediate)
G_cycle = nx.cycle_graph(6)
A_cycle = nx.adjacency_matrix(G_cycle).toarray().astype(float)
print("\n--- Cycle Graph C_6 ---")
verify_cheeger_inequality(A_cycle)

---
## Example 10: Spectral Embedding

**Spectral embedding** uses Laplacian eigenvectors to embed graph nodes into Euclidean space.

### Laplacian Eigenmaps

Goal: Find low-dimensional embedding $Y$ that preserves graph structure.

$$\min_Y \sum_{ij} A_{ij} \|y_i - y_j\|^2 = \min_Y \text{tr}(Y^T L Y)$$

subject to $Y^T D Y = I$.

Solution: Use eigenvectors $u_2, u_3, \ldots, u_{d+1}$ corresponding to smallest non-zero eigenvalues.

In [ ]:
def spectral_embedding(A: np.ndarray, dim: int = 2) -> np.ndarray:
    """
    Compute spectral embedding using Laplacian Eigenmaps.
    
    Args:
        A: Adjacency matrix
        dim: Embedding dimension
    
    Returns:
        Embedding matrix (n x dim)
    """
    L, _, _ = compute_graph_laplacian(A)
    eigenvalues, eigenvectors = la.eigh(L)
    idx = np.argsort(eigenvalues)
    
    # Skip first eigenvector (constant), take next 'dim' eigenvectors
    embedding = eigenvectors[:, idx[1:dim+1]]
    
    return embedding

# Create a graph with natural 2D structure (grid)
G_grid = nx.grid_2d_graph(5, 5)
G_grid = nx.convert_node_labels_to_integers(G_grid)

# Store original positions
original_pos = {i: (i % 5, i // 5) for i in range(25)}

A_grid = nx.adjacency_matrix(G_grid).toarray().astype(float)
embedding = spectral_embedding(A_grid, dim=2)

print("=== Spectral Embedding of 5x5 Grid ===")
print(f"Embedding shape: {embedding.shape}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original grid layout
axes[0].scatter([p[0] for p in original_pos.values()], 
                [p[1] for p in original_pos.values()], 
                c=range(25), cmap='viridis', s=100)
for (i, j) in G_grid.edges():
    axes[0].plot([original_pos[i][0], original_pos[j][0]], 
                 [original_pos[i][1], original_pos[j][1]], 'gray', alpha=0.3)
axes[0].set_title('Original Grid Layout')
axes[0].set_aspect('equal')

# Spectral embedding
axes[1].scatter(embedding[:, 0], embedding[:, 1], 
                c=range(25), cmap='viridis', s=100)
for (i, j) in G_grid.edges():
    axes[1].plot([embedding[i, 0], embedding[j, 0]], 
                 [embedding[i, 1], embedding[j, 1]], 'gray', alpha=0.3)
axes[1].set_title('Spectral Embedding (Laplacian Eigenmaps)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

# The structure is preserved despite using only spectral information!
print("\nNote: Spectral embedding recovers the grid structure from connectivity alone!")

---
## Example 11: Graph Partitioning

**Graph partitioning** divides a graph into parts while minimizing cut edges.

### Objectives

1. **Minimum Cut**: $\min_S |\partial S|$
   - Can lead to unbalanced partitions

2. **Ratio Cut**: $\min_S \frac{|\partial S|}{|S| \cdot |\bar{S}|}$
   - Encourages balanced partitions

3. **Normalized Cut**: $\min_S \frac{|\partial S|}{\text{vol}(S)} + \frac{|\partial S|}{\text{vol}(\bar{S})}$
   - where $\text{vol}(S) = \sum_{i \in S} d_i$

### Spectral Relaxation
The Fiedler vector provides an approximate solution to these NP-hard problems.

In [ ]:
def graph_partition_metrics(A: np.ndarray, S: set) -> Dict[str, float]:
    """
    Compute various partition quality metrics.
    """
    n = len(A)
    S_bar = set(range(n)) - S
    
    # Cut size
    cut = sum(A[i, j] for i in S for j in S_bar)
    
    # Volumes
    degrees = A.sum(axis=1)
    vol_S = sum(degrees[i] for i in S)
    vol_S_bar = sum(degrees[i] for i in S_bar)
    
    # Metrics
    ratio_cut = cut / (len(S) * len(S_bar)) if len(S) > 0 and len(S_bar) > 0 else float('inf')
    normalized_cut = (cut / vol_S + cut / vol_S_bar) if vol_S > 0 and vol_S_bar > 0 else float('inf')
    
    return {
        'cut_size': cut,
        'ratio_cut': ratio_cut,
        'normalized_cut': normalized_cut,
        'balance': min(len(S), len(S_bar)) / max(len(S), len(S_bar))
    }

def spectral_partition(A: np.ndarray, method: str = 'fiedler') -> Tuple[set, set]:
    """
    Partition graph using spectral methods.
    """
    L, _, _ = compute_graph_laplacian(A)
    eigenvalues, eigenvectors = la.eigh(L)
    idx = np.argsort(eigenvalues)
    
    fiedler = eigenvectors[:, idx[1]]
    
    # Partition by sign
    S = set(np.where(fiedler <= 0)[0])
    S_bar = set(np.where(fiedler > 0)[0])
    
    return S, S_bar

# Create a graph with two communities
np.random.seed(42)
G = nx.planted_partition_graph(2, 8, 0.6, 0.05)  # 2 groups of 8, high internal, low external
A = nx.adjacency_matrix(G).toarray().astype(float)

# Ground truth partition
true_S = set(range(8))
true_S_bar = set(range(8, 16))

# Spectral partition
pred_S, pred_S_bar = spectral_partition(A)

print("=== Graph Partitioning ===")
print(f"Ground truth: S = {sorted(true_S)}, S̄ = {sorted(true_S_bar)}")
print(f"Spectral:     S = {sorted(pred_S)}, S̄ = {sorted(pred_S_bar)}")

# Compute metrics
true_metrics = graph_partition_metrics(A, true_S)
pred_metrics = graph_partition_metrics(A, pred_S)

print("\n--- Partition Metrics ---")
print(f"{'Metric':<20} {'Ground Truth':<15} {'Spectral':<15}")
print("-" * 50)
for key in true_metrics:
    print(f"{key:<20} {true_metrics[key]:<15.4f} {pred_metrics[key]:<15.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

pos = nx.spring_layout(G, seed=42)

# Ground truth
colors_true = ['red' if i in true_S else 'blue' for i in range(16)]
nx.draw(G, pos, node_color=colors_true, with_labels=True, ax=axes[0])
axes[0].set_title(f'Ground Truth (cut={true_metrics["cut_size"]:.0f})')

# Spectral partition
colors_pred = ['red' if i in pred_S else 'blue' for i in range(16)]
nx.draw(G, pos, node_color=colors_pred, with_labels=True, ax=axes[1])
axes[1].set_title(f'Spectral Partition (cut={pred_metrics["cut_size"]:.0f})')

plt.tight_layout()
plt.show()

---
## Example 12: Commute Time and Resistance Distance

**Commute time** and **resistance distance** are fundamental concepts connecting random walks and electrical networks.

### Definitions

- **Hitting time** $H(i,j)$: Expected steps to reach $j$ starting from $i$
- **Commute time** $C(i,j) = H(i,j) + H(j,i)$: Expected round-trip time
- **Resistance distance**: $R_{ij} = (e_i - e_j)^T L^+ (e_i - e_j)$ where $L^+$ is Moore-Penrose pseudoinverse

### Key Relationship

$$C(i,j) = 2m \cdot R_{ij}$$

where $m$ is the number of edges.

In [ ]:
def compute_resistance_distance(L: np.ndarray) -> np.ndarray:
    """
    Compute resistance distance matrix using Laplacian pseudoinverse.
    
    R_ij = L^+_{ii} + L^+_{jj} - 2*L^+_{ij}
    """
    n = len(L)
    L_pinv = np.linalg.pinv(L)
    
    R = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            R[i, j] = L_pinv[i, i] + L_pinv[j, j] - 2 * L_pinv[i, j]
    
    return R

def compute_commute_time(A: np.ndarray) -> np.ndarray:
    """
    Compute commute time matrix.
    
    C(i,j) = 2m * R_ij
    """
    L, _, _ = compute_graph_laplacian(A)
    R = compute_resistance_distance(L)
    m = A.sum() / 2  # Number of edges
    
    return 2 * m * R

def simulate_hitting_time(A: np.ndarray, start: int, target: int, 
                          n_simulations: int = 10000) -> float:
    """
    Estimate hitting time via Monte Carlo simulation.
    """
    n = len(A)
    degrees = A.sum(axis=1)
    
    total_steps = 0
    for _ in range(n_simulations):
        current = start
        steps = 0
        while current != target:
            # Random walk step
            neighbors = np.where(A[current] > 0)[0]
            current = np.random.choice(neighbors)
            steps += 1
            if steps > 10000:  # Prevent infinite loop
                break
        total_steps += steps
    
    return total_steps / n_simulations

# Create a small graph
G = nx.cycle_graph(6)
# Add some extra edges for variety
G.add_edge(0, 3)
A = nx.adjacency_matrix(G).toarray().astype(float)

print("=== Commute Time and Resistance Distance ===")
print("Graph: Cycle C_6 with extra edge (0,3)\n")

L, _, _ = compute_graph_laplacian(A)
R = compute_resistance_distance(L)
C = compute_commute_time(A)

print("Resistance Distance Matrix R:")
print(np.round(R, 4))

print("\nCommute Time Matrix C = 2m * R:")
print(np.round(C, 2))

# Verify with simulation
print("\n--- Monte Carlo Verification ---")
np.random.seed(42)
pairs_to_test = [(0, 1), (0, 3), (0, 4)]

for i, j in pairs_to_test:
    H_ij = simulate_hitting_time(A, i, j, n_simulations=5000)
    H_ji = simulate_hitting_time(A, j, i, n_simulations=5000)
    C_simulated = H_ij + H_ji
    C_analytical = C[i, j]
    
    print(f"C({i},{j}): Analytical = {C_analytical:.2f}, Simulated = {C_simulated:.2f}")

# Visualize resistance distance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', 
        node_size=500, ax=axes[0])
axes[0].set_title('Graph Structure')

im = axes[1].imshow(R, cmap='YlOrRd')
axes[1].set_title('Resistance Distance Matrix')
axes[1].set_xlabel('Node j')
axes[1].set_ylabel('Node i')
plt.colorbar(im, ax=axes[1], label='R_ij')

plt.tight_layout()
plt.show()

print("\n=== Key Insight ===")
print("Resistance distance captures graph connectivity:")
print(" - Low R_ij: Multiple paths between i and j (well-connected)")
print(" - High R_ij: Few paths between i and j (bottleneck)")

---
## Summary

This notebook covered 12 fundamental topics in spectral graph theory:

| Topic | Key Takeaway |
|-------|-------------|
| **Graph Laplacian** | $L = D - A$, captures local structure |
| **Normalized Laplacian** | Bounded spectrum $[0, 2]$, better for irregular graphs |
| **Eigenvalues** | Encode structural properties (connectivity, cuts) |
| **Algebraic Connectivity** | $\lambda_2 > 0$ iff connected |
| **Spectral Clustering** | Eigenvectors embed nodes for clustering |
| **Random Walks** | $P = D^{-1}A$, stationary distribution $\pi_i \propto d_i$ |
| **Graph Fourier Transform** | $\hat{x} = U^T x$, spectral decomposition of signals |
| **Graph Signal Processing** | Filtering via $g(L)$ |
| **Cheeger Inequality** | $\lambda_2/2 \le h(G) \le \sqrt{2\lambda_2}$ |
| **Spectral Embedding** | Laplacian eigenvectors preserve structure |
| **Graph Partitioning** | Fiedler vector approximates optimal cuts |
| **Commute Time/Resistance** | $C(i,j) = 2m \cdot R_{ij}$ |

These concepts form the mathematical foundation for **Graph Neural Networks** (GNNs), where spectral filters (like in GCN, ChebNet) operate on graph-structured data.